# pkg-halluc on Kaggle

Thin wrapper around the `pkg_halluc` CLI -- all the actual pipeline logic
lives in the repo (`pkg_halluc/`), not in this notebook. See the repo's
`README.md` for what each stage does; this notebook just runs them in order.

**Getting the code onto Kaggle -- two options (cell below handles both):**
- **Already pushed to GitHub:** paste the URL into `REPO_URL` in the next cell.
- **Not on GitHub yet:** zip this project folder yourself (the one containing
  `pyproject.toml`) and upload it as a new Kaggle Dataset -- exactly the same
  way you already upload `Adaptive-Unlearning-952E.zip` -- then attach it
  here via **Add Input**. Leave `REPO_URL = None` and the next cell finds it
  automatically.

**Before running:** in the Kaggle notebook editor, enable **GPU** and
**Internet** under Settings. If you'd rather not fetch the Adaptive
Unlearning source over the network each session, attach it as a private
Kaggle Dataset via **Add Input** too -- `fetch-deps` below finds it the same
way (see README.md "Fetching the third-party code").

## 1. Get the code

In [ ]:
# Option 1: already pushed to GitHub? Paste the URL below and this clones it.
# Option 2 (no GitHub repo yet): leave REPO_URL = None. Instead, zip the
# project folder yourself and upload it as a Kaggle Dataset -- exactly the
# same way you already did for Adaptive-Unlearning-952E.zip -- then attach
# it here via "Add Input". This cell finds it automatically, no URL needed.
REPO_URL = None  # e.g. "https://github.com/<you>/<repo>.git"

import glob
import os
import shutil
import zipfile

PROJECT_DIR = "/kaggle/working/pkg-halluc"


def _looks_like_project(d):
    return os.path.isfile(os.path.join(d, "pyproject.toml")) and os.path.isdir(
        os.path.join(d, "pkg_halluc")
    )


if _looks_like_project(PROJECT_DIR):
    print("Already present at", PROJECT_DIR)
elif REPO_URL:
    !git clone --depth 1 "$REPO_URL" "$PROJECT_DIR"
else:
    zip_candidates = glob.glob("/kaggle/input/**/pkg-halluc.zip", recursive=True) or glob.glob(
        "/kaggle/input/**/*.zip", recursive=True
    )
    dir_candidates = [
        os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    ]
    assert zip_candidates or dir_candidates, (
        "Couldn't find the project under /kaggle/input, and REPO_URL is empty. "
        "Zip the project folder (the one containing pyproject.toml) on your machine, "
        "upload it as a new Kaggle Dataset, then attach it to this notebook via "
        "'Add Input' and re-run this cell."
    )
    if zip_candidates:
        print("Found zip:", zip_candidates[0])
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(PROJECT_DIR)
        # if the zip has one wrapping folder (e.g. "project/"), hoist it up
        entries = [e for e in os.listdir(PROJECT_DIR) if not e.startswith(".")]
        if len(entries) == 1 and os.path.isdir(os.path.join(PROJECT_DIR, entries[0])):
            inner = os.path.join(PROJECT_DIR, entries[0])
            if _looks_like_project(inner):
                for item in os.listdir(inner):
                    shutil.move(os.path.join(inner, item), os.path.join(PROJECT_DIR, item))
                os.rmdir(inner)
    else:
        print("Found pre-extracted dataset:", dir_candidates[0])
        shutil.copytree(dir_candidates[0], PROJECT_DIR, dirs_exist_ok=True)

assert _looks_like_project(PROJECT_DIR), f"{PROJECT_DIR} doesn't look like the project (missing pyproject.toml)"
%cd {PROJECT_DIR}

## 2. Install

In [ ]:
# NOTE: plain (non-editable) install on purpose. `pip install -e .` registers
# the package via a .pth file that Python only reads at interpreter startup --
# since this kernel is already running, editable installs done mid-session
# aren't importable until you restart the kernel. A regular install copies
# files straight into site-packages, which a running kernel *does* pick up
# immediately. (If you're editing pkg_halluc's source and want changes to
# take effect without restarting, use -e and restart the kernel after.)
!pip install . -q
!pip show pkg_halluc

In [ ]:
import pkg_halluc, torch
print("pkg_halluc", pkg_halluc.__version__, "from", pkg_halluc.__file__)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- enable GPU in Settings!")
# If this cell fails with ModuleNotFoundError even though the cell above
# printed a real version for `pip show pkg_halluc`, restart the kernel
# (Run > Restart session) and re-run from the top -- the install cell above
# doesn't need to be re-run, pip already did its job.

## 3. Pick a config

`configs/smoke_test.json` mirrors the tiny settings this pipeline was first
validated with (fast, not meant for real numbers). Switch to
`configs/default.json` (or your own, see README.md "Configuration") for an
actual run.

In [ ]:
CONFIG = "configs/smoke_test.json"
!cat {CONFIG}

## 4. Run the pipeline, stage by stage

Each cell is independently re-runnable -- if a Kaggle session gets cut off
mid-training, restart and re-run from wherever you left off instead of from
the top (everything under `/kaggle/working` persists for the session; fetched
deps / the downloaded model / already-built data are skipped automatically on
re-run).

In [ ]:
!pkg_halluc fetch-deps --config {CONFIG}

In [ ]:
!pkg_halluc download-model --config {CONFIG}

In [ ]:
!pkg_halluc build-data --config {CONFIG}

In [ ]:
!pkg_halluc train --config {CONFIG} --method all

In [ ]:
!pkg_halluc evaluate --config {CONFIG} --method all

## 5. Report

In [ ]:
!pkg_halluc report --config {CONFIG}

In [ ]:
import pandas as pd
display(pd.read_csv("/kaggle/working/outputs/table1_hallucination_rate.csv"))
display(pd.read_csv("/kaggle/working/outputs/table1_by_context.csv"))